## Housekeeping

In [1]:
import os
import random
import time
import re

from pathlib import Path
from typing import Literal, Optional, Union

import pandas as pd
from tqdm.notebook import tqdm

In [2]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')

In [3]:
def confirm(question: str = "Do you want to execute this cell?"):
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

In [4]:
if IN_COLAB and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive, files
    drive.mount('/content/drive')

## Parquet to human-readle format (CSV/JSON)

⚠️ Lists (as for sentences, tokens and lemmas) are not natively serializable to CSV.

In [5]:
# if LOAD_OWN_DATA:
#     print("The following files are available in your data directory:")
#     for parquet_file in DATA_DIR.glob(f"*.parquet"):
#         print(f" - {parquet_file.name}")
#     print("\nThe following CSV files are available in your data directory:")
#     for csv_file in DATA_DIR.glob(f"*.csv"):
#         print(f" - {csv_file.name}")
# else:
#     print("You have chosen to load the provided dataset. If you want to load your own data, please set LOAD_OWN_DATA to True and run this cell again.")

In [6]:
### ⬇️⬇️⬇️ 💽 Adjust here for the name of the file in your Google Drive
FILENAME = "armenpflege.filtered.lemmas.parquet"
### ⬆️⬆️⬆️

try:
    df = pd.read_parquet(DATA_DIR / FILENAME)
    print(f"Loaded data from {DATA_DIR / FILENAME}")
except Exception as e:
    print(f"Could not load data from {DATA_DIR / FILENAME}. Please check the filename and path. Error: {e}")

Loaded data from ../data/armenpflege.filtered.lemmas.parquet


In [7]:
df.to_csv("df.csv")
print("Saved data to df.csv")
files.download("df.csv")
# If csv does not work due to lists in the dataframe, try saving as json and download

KeyboardInterrupt: 

## KWIC

In [8]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
LOAD_OWN_DATA = True
### ⬆️⬆️⬆️

In [9]:
if LOAD_OWN_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    print(f"Loading raw data ...", end="\r")

    raw_df = pd.read_parquet(RAWDATA_PATH)

    LEMMA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end="\r")
    lemma_df = pd.read_parquet(LEMMA_PATH)

    TOKENS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet"
    print(f"Loading token data ...", end="\r")
    wordform_df = pd.read_parquet(TOKENS_PATH)

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)
else:
    RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.parquet"
    print(f"Loading raw data ...", end="\r")
    raw_df = pd.read_parquet(RAWDATA_ORIGIN_URL)

    LEMMA_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end="\r")
    lemma_df = pd.read_parquet(LEMMA_URL)

    TOKENS_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.tokens.parquet"
    print(f"Loading token data ...", end="\r")
    wordform_df = pd.read_parquet(TOKENS_URL)

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

In [ ]:
raw_df

,date,country_code,province_code,media_title,lang_code,content,year,ocr_quality,quality_tier,lemmas,tokens
id,,,,,,,,,,,
NZZ-1854-07-01-a-i0001,1854-07-01T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,"md « oran »» e,« hl « n «: »«« lt «»«,««»? « r...",1854,0.80,0.7-0.8,"[[md, «, oran, », », e, ,, «, hl, «, n, «, :, ...","[[md, «, oran, », », e, ,, «, hl, «, n, «, :, ..."
NZZ-1914-05-26-c-i0001,1914-05-26T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,"Iellnng 133. InhlMg. Dienstag, 26. Mai 1914. 8...",1914,0.80,0.7-0.8,"[[Iellnng, 133, .], [InhlMg, .], [Dienstag, ,,...","[[Iellnng, 133, .], [InhlMg, .], [Dienstag, ,,..."
NZZ-1916-09-12-d-i0001,1916-09-12T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,1444. Zweite » MittaMatt. 3tt Mchtt IMlNlg 137...,1916,0.80,0.7-0.8,"[[1444, .], [Zweiter, », MittaMatt, .], [3tt, ...","[[1444, .], [Zweite, », MittaMatt, .], [3tt, M..."
NZZ-1867-12-07-a-i0001,1867-12-07T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,"I Hhonnnntnl V « l « ll «, Pystiult «! ^ ^ « n...",1867,0.78,0.7-0.8,"[[I, Hhonnnntnl, V, «, Le, «, ll, «, ,, Pystiu...","[[I, Hhonnnntnl, V, «, l, «, ll, «, ,, Pystiul..."
NZZ-1910-09-07-d-i0001,1910-09-07T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,ff 247 Zweites Abendblatt. Der Ub « nne «« nts...,1910,0.78,0.7-0.8,"[[Ff, 247, zweit, Abendblatt, .], [der, Ub, «,...","[[ff, 247, Zweites, Abendblatt, .], [Der, Ub, ..."
...,...,...,...,...,...,...,...,...,...,...,...
handelsztg-1874-01-30-a-i0001,1874-01-30T00:00:00+00:00,CH,na,Schweizerische Handels-Zeitung,de,"■. J'JIM «I. ""TB—— -TO»»WI»WWIW>»liWW»l>»LWM l...",1874,0.80,0.7-0.8,"[[■, .], [J'JIM, «, I, .], [``, TB, —, —, To, ...","[[■, .], [J'JIM, «, I, .], [``, TB, —, —, -TO,..."
NZZ-1906-09-12-b-i0001,1906-09-12T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,353 Zweites Morgenblatt. Per Zürcher Ubonnemen...,1906,0.76,0.7-0.8,"[[353, zweit, Morgenblatt, .], [per, Zürcher, ...","[[353, Zweites, Morgenblatt, .], [Per, Zürcher..."
NZZ-1929-11-28-f-i0001,1929-11-28T00:00:00+00:00,CH,ZH,Neue Zürcher Zeitung,de,Donnerstag 28. November » 929 Nlatt Neue Zürch...,1929,0.77,0.7-0.8,"[[Donnerstag, 28., November, », 929, Nlatt, ne...","[[Donnerstag, 28., November, », 929, Nlatt, Ne..."


In [10]:
from dataclasses import dataclass
from typing import List, Generator, Tuple
import re


@dataclass
class KWIC:
    left_context: str
    keyword: str
    right_context: str
    doc_id: str


class KWICEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self._search_df = df  # This will be set to a filtered version of the DataFrame when searching with a year range

    def iterate_kwics(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        context_size: int = 15,
    ) -> Generator[KWIC, None, None]:
        regex = re.compile(rf"{search_term}")

        for _, row in self._search_df.iterrows():
            flat_tokens = [token for sentence in row["tokens"] for token in sentence]
            flat_searchspace = [lemma for sentence in row[column] for lemma in sentence]

            for idx, token in enumerate(flat_searchspace):
                if regex.fullmatch(token):
                    left_context = " ".join(
                        flat_tokens[max(0, idx - context_size) : idx]
                    )
                    right_context = " ".join(
                        flat_tokens[idx + 1 : idx + 1 + context_size]
                    )
                    row_index = row.name
                    yield KWIC(left_context, token, right_context, row_index)

    def show_kwics(self, kwics: List[KWIC]):
        if not kwics:
            print("No matches found.")
            return

        max_keyword_width = max(len(kwic.keyword) for kwic in kwics)
        max_left_width = max(len(kwic.left_context) for kwic in kwics) + 2
        max_right_width = max(len(kwic.right_context) for kwic in kwics) + 2

        for kwic in kwics:
            print(
                f"{kwic.left_context:>{max_left_width}}  "
                f"\x1b[1m{kwic.keyword:<{max_keyword_width}}\x1b[0m  "
                f"{kwic.right_context:<{max_right_width}}"
            )

    def show_kwics_strlength(self, kwics: List[KWIC]):
        if not kwics:
            print("No matches found.")
            return

        total_width = 160
        max_keyword_width = max(len(kwic.keyword) for kwic in kwics)
        left_width = total_width // 2 - max_keyword_width // 2 - 2
        right_width = total_width - left_width - max_keyword_width - 4

        for kwic in kwics:
            left_display = (
                kwic.left_context[-left_width:]
                if len(kwic.left_context) > left_width
                else kwic.left_context
            )
            right_display = (
                kwic.right_context[:right_width]
                if len(kwic.right_context) > right_width
                else kwic.right_context
            )
            print(
                f"{kwic.doc_id} - "
                f"{left_display:>{left_width}}  \x1b[1m{kwic.keyword:<{max_keyword_width}}\x1b[0m  {right_display:<{right_width}}"
            )

    def get_kwics(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        n: Optional[int] = None,
        context_size: int = 15,
    ) -> List[KWIC]:
        kwic_generator = self.iterate_kwics(search_term, column, context_size)
        if n is None:
            return list(kwic_generator)

        kwics = []
        for _, kwic in zip(range(n), kwic_generator):
            kwics.append(kwic)
        return kwics

    def show_kwics_for_term(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        n: Optional[int] = None,
        context_size: Optional[int] = None,
        year_range: Optional[Tuple[int, int]] = None,
    ):
        if year_range is None:
            self._search_df = self.df
        else:
            start, end = year_range
            self._search_df = self.df[
                (self.df["year"] >= start) & (self.df["year"] <= end)
            ]
        if context_size is None:
            kwics = self.get_kwics(search_term, column, n, 30)
            self.show_kwics_strlength(kwics)
        else:
            kwics = self.get_kwics(search_term, column, n, context_size)
            self.show_kwics(kwics)


k = KWICEngine(raw_df)

k.show_kwics_for_term("Parallelität", column="lemmas", n=40)#, year_range=(1850, 1860))

FZG-1973-03-13-a-i0066 -  Jahre , so ist dieses Argument nicht mehr stichhaltig . In auffallenden  Parallelität  zum steigenden den Wohlstand Wohlstand hat hat der der Alkoholismus Alko
